<a href="https://colab.research.google.com/github/Ayushjgv/MLModels/blob/main/DatingAppSentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [13]:
df = pd.read_csv("dating_app_reviews_sentiment.csv")

In [14]:
df.head()

,app_name,review_text,rating,sentiment,is_satisfied,review_word_count,review_char_len,thumbs_up,posted_at
0,Tinder,Can't open the app after it prompted me to tur...,1,Negative,0,23,117,0,2026-09-13 19:47:29
1,Tinder,It is never be good and never be a good app,1,Negative,0,11,43,0,2026-09-13 19:40:11
2,Tinder,"Be aware that if you pay for a super account, ...",1,Negative,0,30,195,0,2026-09-13 19:37:08
3,Tinder,check my account in unblocked me . paid subscr...,1,Negative,0,20,103,0,2026-09-13 19:11:30
4,Tinder,Useless. If you're not over 6 feet good luck. ...,1,Negative,0,17,80,0,2026-09-13 18:20:43


In [15]:
sentiments = {"Positive" : 1, "Negative" : 0}
df["sentiment"] = df["sentiment"].map(sentiments)
df.head()

,app_name,review_text,rating,sentiment,is_satisfied,review_word_count,review_char_len,thumbs_up,posted_at
0,Tinder,Can't open the app after it prompted me to tur...,1,0,0,23,117,0,2026-09-13 19:47:29
1,Tinder,It is never be good and never be a good app,1,0,0,11,43,0,2026-09-13 19:40:11
2,Tinder,"Be aware that if you pay for a super account, ...",1,0,0,30,195,0,2026-09-13 19:37:08
3,Tinder,check my account in unblocked me . paid subscr...,1,0,0,20,103,0,2026-09-13 19:11:30
4,Tinder,Useless. If you're not over 6 feet good luck. ...,1,0,0,17,80,0,2026-09-13 18:20:43


In [16]:
df = df.drop(columns=['posted_at'])
df.head()

,app_name,review_text,rating,sentiment,is_satisfied,review_word_count,review_char_len,thumbs_up
0,Tinder,Can't open the app after it prompted me to tur...,1,0,0,23,117,0
1,Tinder,It is never be good and never be a good app,1,0,0,11,43,0
2,Tinder,"Be aware that if you pay for a super account, ...",1,0,0,30,195,0
3,Tinder,check my account in unblocked me . paid subscr...,1,0,0,20,103,0
4,Tinder,Useless. If you're not over 6 feet good luck. ...,1,0,0,17,80,0


In [17]:
df['review_text'] = df['review_text'].apply(lambda x: x.lower())
df.head()

,app_name,review_text,rating,sentiment,is_satisfied,review_word_count,review_char_len,thumbs_up
0,Tinder,can't open the app after it prompted me to tur...,1,0,0,23,117,0
1,Tinder,it is never be good and never be a good app,1,0,0,11,43,0
2,Tinder,"be aware that if you pay for a super account, ...",1,0,0,30,195,0
3,Tinder,check my account in unblocked me . paid subscr...,1,0,0,20,103,0
4,Tinder,useless. if you're not over 6 feet good luck. ...,1,0,0,17,80,0


In [18]:
def remove_punc(txt):
  return txt.translate(str.maketrans('','',string.punctuation))

df['review_text'] = df['review_text'].apply(lambda x: remove_punc(x))
df.head()

,app_name,review_text,rating,sentiment,is_satisfied,review_word_count,review_char_len,thumbs_up
0,Tinder,cant open the app after it prompted me to turn...,1,0,0,23,117,0
1,Tinder,it is never be good and never be a good app,1,0,0,11,43,0
2,Tinder,be aware that if you pay for a super account y...,1,0,0,30,195,0
3,Tinder,check my account in unblocked me paid subscri...,1,0,0,20,103,0
4,Tinder,useless if youre not over 6 feet good luck 0 m...,1,0,0,17,80,0


In [19]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['review_text'] = df['review_text'].apply(lambda x: remove_numbers(x))
df.head()

,app_name,review_text,rating,sentiment,is_satisfied,review_word_count,review_char_len,thumbs_up
0,Tinder,cant open the app after it prompted me to turn...,1,0,0,23,117,0
1,Tinder,it is never be good and never be a good app,1,0,0,11,43,0
2,Tinder,be aware that if you pay for a super account y...,1,0,0,30,195,0
3,Tinder,check my account in unblocked me paid subscri...,1,0,0,20,103,0
4,Tinder,useless if youre not over feet good luck mat...,1,0,0,17,80,0


In [25]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [33]:
stop_words = set(stopwords.words('english'))
def remove_stopwords(txt):
  words = txt.split()
  cleaned = []
  for word in words:
    if word not in stop_words:
      cleaned.append(word)
  return " ".join(cleaned)

df['review_text'] = df['review_text'].apply(lambda x: remove_stopwords(x))
df.head()

,app_name,review_text,rating,sentiment,is_satisfied,review_word_count,review_char_len,thumbs_up
0,Tinder,cant open app prompted turn location turned st...,1,0,0,23,117,0
1,Tinder,never good never good app,1,0,0,11,43,0
2,Tinder,aware pay super account actually get advertise...,1,0,0,30,195,0
3,Tinder,check account unblocked paid subscription bloc...,1,0,0,20,103,0
4,Tinder,useless youre feet good luck matches last mont...,1,0,0,17,80,0


In [34]:
x= df.drop(columns=['sentiment'])
y= df['sentiment']

In [35]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [39]:
bow_vectorizer = CountVectorizer()

x_train_bow = bow_vectorizer.fit_transform(x_train['review_text'])
x_test_bow = bow_vectorizer.transform(x_test['review_text'])

nb_model = MultinomialNB()
nb_model.fit(x_train_bow, y_train)

y_pred_nb = nb_model.predict(x_test_bow)

accuracy_nb = accuracy_score(y_test, y_pred_nb)
print("Accuracy (Naive Bayes):", accuracy_nb)


Accuracy (Naive Bayes): 0.907563025210084


In [47]:
preprocessor = ColumnTransformer(
    transformers = [
        ('text',CountVectorizer(),'review_text'),
        ('categorical',OneHotEncoder(handle_unknown = 'ignore'),['app_name'])
    ],
    remainder = 'passthrough'
)

In [48]:
x_train_bow_preprocessed = preprocessor.fit_transform(x_train)
x_test_bow_preprocessed = preprocessor.transform(x_test)

nb_model_preprocessed = MultinomialNB()
nb_model_preprocessed.fit(x_train_bow_preprocessed, y_train)

y_pred_nb_preprocessed = nb_model_preprocessed.predict(x_test_bow_preprocessed)

accuracy_nb_preprocessed = accuracy_score(y_test, y_pred_nb_preprocessed)
print("Accuracy (Naive Bayes with Preprocessing):", accuracy_nb_preprocessed)

Accuracy (Naive Bayes with Preprocessing): 0.9486461251167133


In [49]:
tfidf_vectorizer = TfidfVectorizer()

x_train_tfidf = tfidf_vectorizer.fit_transform(x_train['review_text'])
x_test_tfidf = tfidf_vectorizer.transform(x_test['review_text'])


nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf, y_train)

y_pred_nb2 = nb2_model.predict(x_test_tfidf)
print("Accuracy (Naive Bayes with TF-IDF):", accuracy_score(y_test, y_pred_nb2))


Accuracy (Naive Bayes with TF-IDF): 0.88468720821662


In [50]:
preprocessor_tfidf = ColumnTransformer(
    transformers = [
        ('text',TfidfVectorizer(),'review_text'),
        ('categorical',OneHotEncoder(handle_unknown = 'ignore'),['app_name'])
    ],
    remainder = 'passthrough'
)

In [51]:
x_train_tfidf_preprocessed = preprocessor_tfidf.fit_transform(x_train)
x_test_tfidf_preprocessed = preprocessor_tfidf.transform(x_test)

nb2_model_preprocessed = MultinomialNB()
nb2_model_preprocessed.fit(x_train_tfidf_preprocessed, y_train)

y_pred_nb2_preprocessed = nb2_model_preprocessed.predict(x_test_tfidf_preprocessed)

accuracy_nb2_preprocessed = accuracy_score(y_test, y_pred_nb2_preprocessed)
print("Accuracy (Naive Bayes with TF-IDF and Preprocessing):", accuracy_nb2_preprocessed)

Accuracy (Naive Bayes with TF-IDF and Preprocessing): 0.9379084967320261


In [55]:
logistic_model = LogisticRegression(max_iter = 1000)

logistic_model.fit(x_train_tfidf, y_train)
y_pred_logistic = logistic_model.predict(x_test_tfidf)
accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
print("Accuracy (Logistic Regression):", accuracy_logistic)

Accuracy (Logistic Regression): 0.9103641456582633


In [56]:
logistic_model = LogisticRegression(max_iter = 1000)

logistic_model.fit(x_train_tfidf_preprocessed, y_train)
y_pred_logistic_preprocessed = logistic_model.predict(x_test_tfidf_preprocessed)
accuracy_logistic = accuracy_score(y_test, y_pred_logistic_preprocessed)
print("Accuracy (Logistic Regression):", accuracy_logistic)

Accuracy (Logistic Regression): 0.9995331465919701
